# Gateway Routing and Resilience Lab

## The brief: make Riverside's control-plane rules executable

You already decided the order: normalize, check a complete cache, enforce provider-facing limits, reserve budget, filter eligibility, route, recover, settle, and emit telemetry. Now you will make every decision visible under deterministic traffic.

> **What you finished last time:** [Gateway Control-Plane Theory](01-gateway-control-plane-theory.ipynb) separated policy from serving and retry from fallback.
> **What this notebook delivers:** a 75-90 minute executable gateway lab with deterministic providers, adapters, routing, limits, budget accounting, caches, recovery, telemetry, and cost/latency reports.
> **Fresh-kernel contract:** no state is shared with the theory notebook. The setup cell below recreates the small Riverside fixture and verifies it.

| Part | Failure first | Measured unlock |
|---|---|---|
| 1. Providers and adapters | Native shapes leak | One normalized contract |
| 2. Eligibility and routing | Cheapest can be illegal; equal turns ignore work | Approved-set routing including least-busy |
| 3. Limits and budget | One request counter misses tokens, concurrency, and spend | Multi-axis admission plus reserve/settle |
| 4. Cache | Exact keys go stale; similarity confuses negation | Versioned exact reuse and unsafe semantic evidence |
| 5. Recovery | Blind retries multiply work | Error-aware retry then policy-safe fallback |
| 6. Telemetry | "Success" hides the attempt story | Request/attempt records and cost/latency report |

**Boundaries:** no network call is required. Circuit breakers and serving backpressure remain forward links because this lab does not measure them.

In [ ]:
# Setup - Recreate and verify the fresh-kernel lab environment
from __future__ import annotations

from collections import Counter, deque
from dataclasses import asdict, dataclass, field
from hashlib import sha256
from typing import Any, Callable
import json
import math
import os
import random
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
rng = random.Random(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "font.size": 10})

RIVERSIDE_PROMPT = "How many chapters are in The Cartographer's Cipher?"
LOCAL_MODEL_ALIAS = "local-llama-8b"
LOCAL_MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"

print(f"seed: {SEED}")
print(f"running prompt: {RIVERSIDE_PROMPT}")
print(f"local routing alias: {LOCAL_MODEL_ALIAS} -> simulated metadata {LOCAL_MODEL_ID}")
print("Fresh-kernel check passed. No transformer is loaded and no answer quality is claimed.")

## 1. Attempt 1: call deterministic providers without adapters

The same question returns three native shapes. Deterministic providers are not production doubles; they hold latency, token use, cost, and failure plans still so you can isolate gateway decisions.

In [ ]:
# Part 1 - Define deterministic native providers with incompatible SDK-shaped responses
@dataclass(frozen=True)
class NativeResult:
    text: str
    input_tokens: int
    output_tokens: int
    latency_ms: float


class DeterministicNativeProvider:
    def __init__(self, name, response_style, latencies_ms, output_tokens, failure_plan=None):
        self.name = name
        self.response_style = response_style
        self.latencies_ms = list(latencies_ms)
        self.output_tokens = list(output_tokens)
        self.failure_plan = list(failure_plan or [])
        self.calls = 0

    def complete(self, prompt):
        index = self.calls
        self.calls += 1
        if index < len(self.failure_plan) and self.failure_plan[index] is not None:
            raise self.failure_plan[index]
        result = NativeResult(
            text="There are 23 chapters.",
            input_tokens=len(prompt.split()),
            output_tokens=self.output_tokens[index % len(self.output_tokens)],
            latency_ms=self.latencies_ms[index % len(self.latencies_ms)],
        )
        if self.response_style == "openai":
            return {
                "choices": [{"message": {"content": result.text}}],
                "usage": {"prompt_tokens": result.input_tokens, "completion_tokens": result.output_tokens},
                "latency_ms": result.latency_ms,
            }
        if self.response_style == "anthropic":
            return {
                "content": [{"text": result.text}],
                "usage": {"input_tokens": result.input_tokens, "output_tokens": result.output_tokens},
                "latency_ms": result.latency_ms,
            }
        return {
            "generated_text": result.text,
            "prompt_tokens": result.input_tokens,
            "generated_tokens": result.output_tokens,
            "latency_ms": result.latency_ms,
        }


native_examples = {
    "openai-shaped": DeterministicNativeProvider("hosted-fast", "openai", [180], [7]).complete(RIVERSIDE_PROMPT),
    "anthropic-shaped": DeterministicNativeProvider("hosted-premium", "anthropic", [280], [8]).complete(RIVERSIDE_PROMPT),
    "local-shaped": DeterministicNativeProvider("local-private", "local", [420], [6]).complete(RIVERSIDE_PROMPT),
}
for name, result in native_examples.items():
    print(f"{name:<18} top-level keys={sorted(result)}")
print("Complaint: callers still need provider-specific extraction logic.")

### Normalize requests, responses, and errors

The adapter owns native SDK syntax. The gateway owns one contract.

| Contract | Required evidence |
|---|---|
| Request | ID, tenant, prompt, token estimate, region, sensitivity, capability, corpus/policy versions |
| Response | route/model, text, input/output tokens, latency, cost, finish status, cache state |
| Error | stable code, retryable flag, route, attempt number |

In [ ]:
# Part 1 - Build the normalized contract and provider adapters
@dataclass(frozen=True)
class GatewayRequest:
    request_id: str
    tenant_id: str
    prompt: str
    estimated_input_tokens: int
    max_output_tokens: int
    region: str = "uk"
    sensitivity: str = "confidential"
    capability: str = "chat"
    corpus_version: str = "corpus-v2"
    policy_version: str = "policy-v3"


@dataclass
class GatewayResponse:
    request_id: str
    provider: str
    model_id: str
    text: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    cost_usd: float
    finish_status: str = "stop"
    cache_status: str = "miss"


class GatewayError(Exception):
    def __init__(self, code, message, retryable):
        super().__init__(message)
        self.code = code
        self.retryable = retryable


@dataclass
class ProviderAdapter:
    name: str
    model_id: str
    native: DeterministicNativeProvider
    cost_per_1k_input: float
    cost_per_1k_output: float
    style: str

    def complete(self, request):
        native = self.native.complete(request.prompt)
        if self.style == "openai":
            text = native["choices"][0]["message"]["content"]
            input_tokens = native["usage"]["prompt_tokens"]
            output_tokens = native["usage"]["completion_tokens"]
        elif self.style == "anthropic":
            text = native["content"][0]["text"]
            input_tokens = native["usage"]["input_tokens"]
            output_tokens = native["usage"]["output_tokens"]
        else:
            text = native["generated_text"]
            input_tokens = native["prompt_tokens"]
            output_tokens = native["generated_tokens"]
        cost = (
            input_tokens * self.cost_per_1k_input
            + output_tokens * self.cost_per_1k_output
        ) / 1000
        return GatewayResponse(
            request_id=request.request_id,
            provider=self.name,
            model_id=self.model_id,
            text=text,
            input_tokens=input_tokens,
            output_tokens=output_tokens,
            latency_ms=native["latency_ms"],
            cost_usd=cost,
        )


request = GatewayRequest("req-contract", "riverside", RIVERSIDE_PROMPT, 9, 32)
adapter_specs = [
    ("hosted-fast", "openai", "fast-v1", 0.00015, 0.00060, 180, 7),
    ("hosted-premium", "anthropic", "premium-v3", 0.00080, 0.00240, 280, 8),
    ("local-private", "local", LOCAL_MODEL_ID, 0.0, 0.0, 420, 6),
]
adapter_examples = []
for name, style, model_id, input_price, output_price, latency, output_count in adapter_specs:
    native = DeterministicNativeProvider(name, style, [latency], [output_count])
    adapter = ProviderAdapter(name, model_id, native, input_price, output_price, style)
    adapter_examples.append(adapter.complete(request))

response_columns = set(asdict(adapter_examples[0]))
same_contract = all(set(asdict(response)) == response_columns for response in adapter_examples)
print(pd.DataFrame(asdict(response) for response in adapter_examples).to_string(index=False))
print(f"\nAll adapters returned the same columns: {same_contract}")

**Code walkthrough**

1. \`DeterministicNativeProvider\` varies native syntax while replaying fixed latency, token, and failure plans.
2. \`GatewayRequest\` carries policy inputs; \`GatewayResponse\` carries billing and attribution evidence.
3. \`ProviderAdapter.complete\` is the only place that knows native field names.
4. Prices apply to measured input and output tokens, not a single generic request count.

**Your turn:** change \`max_output_tokens\` in a later request. The adapter should not change; only admission and reservation policy should.

## 2. Eligibility before routing

**Predict:** the local route costs zero, the fast hosted route costs less than premium, and the request requires a hosted UK route. Which route is cheapest globally? Which routes are actually eligible?

In [ ]:
# Part 2 - Separate hard eligibility from soft routing objectives
@dataclass
class Route:
    name: str
    adapter: ProviderAdapter
    region: str
    capabilities: set[str]
    allows_confidential: bool
    healthy: bool
    in_flight: int = 0
    weight: float = 1.0
    ema_latency_ms: float | None = None

    @property
    def estimated_cost_per_1k(self):
        return self.adapter.cost_per_1k_input + self.adapter.cost_per_1k_output


def build_routes():
    specs = [
        ("external-cheap", "openai", "external-v1", "us", False, 0.00005, 0.00020, 120),
        ("hosted-fast", "openai", "fast-v1", "uk", True, 0.00015, 0.00060, 180),
        ("hosted-premium", "anthropic", "premium-v3", "uk", True, 0.00080, 0.00240, 280),
        ("local-private", "local", LOCAL_MODEL_ID, "on-prem", True, 0.0, 0.0, 420),
    ]
    routes = []
    for name, style, model_id, region, confidential, input_price, output_price, latency in specs:
        native = DeterministicNativeProvider(name, style, [latency], [7])
        adapter = ProviderAdapter(name, model_id, native, input_price, output_price, style)
        routes.append(Route(name, adapter, region, {"chat"}, confidential, True))
    return routes


def eligible_routes(request, routes, require_hosted=False):
    return [
        route
        for route in routes
        if route.healthy
        and request.capability in route.capabilities
        and (request.sensitivity != "confidential" or route.allows_confidential)
        and (not require_hosted or route.region == request.region)
    ]


routes = build_routes()
globally_cheapest = min(routes, key=lambda route: route.estimated_cost_per_1k)
approved = eligible_routes(request, routes, require_hosted=True)
approved_cheapest = min(approved, key=lambda route: route.estimated_cost_per_1k)

print(f"globally cheapest: {globally_cheapest.name}")
print(f"eligible routes:   {[route.name for route in approved]}")
print(f"cheapest eligible: {approved_cheapest.name}")
print("Prediction check: the zero-cost local route is still ineligible when hosted UK service is required.")

### Attempt 2: round-robin ignores unfinished work

Round-robin is fair by dispatch count when routes are comparable. It does not look at in-flight work. Least-busy does, but only if queue state is timely and shared across replicas.

In [ ]:
# Part 2 - Compare round-robin with least-busy under variable service times
def simulate_queue_routing(strategy, service_ticks, route_names):
    queues = {name: [] for name in route_names}
    rows = []
    rr_index = 0
    for step, ticks in enumerate(service_ticks):
        for name in route_names:
            queues[name] = [remaining - 1 for remaining in queues[name] if remaining - 1 > 0]
        before = {name: len(queues[name]) for name in route_names}
        if strategy == "round-robin":
            chosen = route_names[rr_index % len(route_names)]
            rr_index += 1
        elif strategy == "least-busy":
            chosen = min(route_names, key=lambda name: (len(queues[name]), route_names.index(name)))
        else:
            raise ValueError(strategy)
        queues[chosen].append(ticks)
        rows.append({"step": step, "chosen": chosen, "service_ticks": ticks, **{f"{name}_busy": before[name] for name in route_names}})
    return pd.DataFrame(rows)


service_ticks = [5, 1, 1, 5, 4, 1, 5, 1, 4, 1, 5, 1, 4, 1, 5, 1, 4, 1]
route_names = ["hosted-fast", "hosted-premium", "local-private"]
rr_queue = simulate_queue_routing("round-robin", service_ticks, route_names)
lb_queue = simulate_queue_routing("least-busy", service_ticks, route_names)

def assignment_summary(frame):
    counts = frame["chosen"].value_counts().reindex(route_names, fill_value=0)
    selected_busy = [
        row[f"{row['chosen']}_busy"]
        for _, row in frame.iterrows()
    ]
    return counts, max(selected_busy), float(np.mean(selected_busy))


rr_counts, rr_peak, rr_mean = assignment_summary(rr_queue)
lb_counts, lb_peak, lb_mean = assignment_summary(lb_queue)
print("Round-robin assignments:\n", rr_counts.to_string(), sep="")
print("\nLeast-busy assignments:\n", lb_counts.to_string(), sep="")
print(f"\nBusy work ahead of chosen route - round-robin mean={rr_mean:.2f}, peak={rr_peak}")
print(f"Busy work ahead of chosen route - least-busy mean={lb_mean:.2f}, peak={lb_peak}")
print("Payoff: least-busy observes unfinished work instead of preserving equal turns.")

In [ ]:
# Part 2 - Plot measured queue state for static evidence
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for axis, frame, title in zip(
    axes,
    [rr_queue, lb_queue],
    ["Round-robin", "Least-busy"],
):
    for route_name in route_names:
        axis.plot(frame["step"], frame[f"{route_name}_busy"], marker="o", label=route_name)
    axis.set_title(title)
    axis.set_xlabel("dispatch step")
    axis.set_ylabel("in-flight before dispatch")
    axis.legend(fontsize=8)
fig.suptitle("The same service times produce different queue-aware decisions")
plt.tight_layout()
plt.show()

print("Static evidence: least-busy steers around whichever route is carrying unfinished work.")

![Animated least-busy routing under changing queue state](images/least-busy-queue-routing-over-time.gif)

The animation and static plot carry the same claim. The GIF remains readable on GitHub; the plot above is generated from the lab's actual queue trace.

**Boundary:** stale per-process counters can make every gateway replica chase the same apparently idle route. Production least-busy needs shared state, capacity weights, or hysteresis.

### Preserve the other routing evidence: latency, cost, and weights

- **Latency-aware:** recent measurements adapt to changing service, but forced exploration prevents one early winner from monopolizing traffic.
- **Cost-aware:** choose the cheapest **eligible** route, never the globally cheapest route.
- **Weighted:** intentionally split approved traffic for canaries or capacity differences.

In [ ]:
# Part 2 - Measure EMA latency learning, weighted routing, and eligible-set cost routing
latency_sequences = {
    "hosted-fast": [190, 180, 175, 185, 178, 182],
    "hosted-premium": [285, 275, 290, 280, 278, 282],
}
alpha = 0.4
ema = {name: None for name in latency_sequences}
latency_choices = []

for step in range(12):
    unexplored = [name for name, value in ema.items() if value is None]
    chosen = unexplored[0] if unexplored else min(ema, key=ema.get)
    sequence = latency_sequences[chosen]
    observed = sequence[latency_choices.count(chosen) % len(sequence)]
    ema[chosen] = observed if ema[chosen] is None else alpha * observed + (1 - alpha) * ema[chosen]
    latency_choices.append(chosen)

weighted_rng = random.Random(7)
weights = {"hosted-fast": 0.7, "hosted-premium": 0.3}
weighted_choices = weighted_rng.choices(list(weights), weights=list(weights.values()), k=200)
weighted_share = pd.Series(weighted_choices).value_counts(normalize=True).sort_index()

cost_choice = min(approved, key=lambda route: route.estimated_cost_per_1k).name
print(f"EMA latencies after forced exploration: {ema}")
print(f"post-exploration choices: {Counter(latency_choices[2:])}")
print(f"weighted target: {weights}")
print("weighted observed share:\n", weighted_share.round(3).to_string(), sep="")
print(f"cheapest eligible route: {cost_choice}")
print("Complaint that remains: every strategy still needs admission controls before dispatch.")

## 3. Request, token, and concurrency limits

A single RPM counter misses three different overloads:

1. a small number of giant prompts can exhaust token quota;
2. slow requests can fill all concurrency even below RPM;
3. a burst can arrive inside an otherwise acceptable minute.

The gateway estimates before dispatch and reconciles actual usage afterward. These in-memory classes teach mechanics; production counters must be atomic and distributed.

In [ ]:
# Part 3 - Implement deterministic multi-axis admission
class MultiAxisLimiter:
    def __init__(self, rpm, tpm, concurrency):
        self.rpm = rpm
        self.tpm = tpm
        self.concurrency = concurrency
        self.request_times = deque()
        self.token_events = deque()
        self.in_flight = 0

    def _expire(self, now):
        while self.request_times and now - self.request_times[0] >= 60:
            self.request_times.popleft()
        while self.token_events and now - self.token_events[0][0] >= 60:
            self.token_events.popleft()

    def admit(self, now, estimated_tokens):
        self._expire(now)
        used_tokens = sum(tokens for _, tokens in self.token_events)
        if len(self.request_times) >= self.rpm:
            return False, "rpm"
        if used_tokens + estimated_tokens > self.tpm:
            return False, "tpm"
        if self.in_flight >= self.concurrency:
            return False, "concurrency"
        self.request_times.append(now)
        self.token_events.append((now, estimated_tokens))
        self.in_flight += 1
        return True, "admitted"

    def release(self):
        if self.in_flight <= 0:
            raise RuntimeError("release called without an admitted request")
        self.in_flight -= 1


limit_cases = [
    ("normal", 0, 30, 0),
    ("concurrency-full", 1, 30, 2),
    ("token-heavy", 2, 95, 0),
    ("rpm-burst-1", 3, 10, 0),
    ("rpm-burst-2", 4, 10, 0),
    ("rpm-burst-3", 5, 10, 0),
]
limit_rows = []
for label, now, tokens, forced_in_flight in limit_cases:
    limiter = MultiAxisLimiter(rpm=3, tpm=100, concurrency=2)
    limiter.request_times.extend([now - 1] * (3 if label.startswith("rpm") else 0))
    limiter.token_events.extend([(now - 1, 20)] if label == "token-heavy" else [])
    limiter.in_flight = forced_in_flight
    allowed, reason = limiter.admit(now, tokens)
    limit_rows.append({"case": label, "estimated_tokens": tokens, "allowed": allowed, "reason": reason})

limit_evidence = pd.DataFrame(limit_rows)
print(limit_evidence.to_string(index=False))
print("\nMeasured result: RPM, TPM, and concurrency reject different request shapes.")

### Token bucket versus sliding window: the old burst evidence still matters

Multi-axis quotas answer **what dimension** is exhausted. Burst shaping answers **how arrivals may spend capacity over time**.

In [ ]:
# Part 3 - Compare token bucket and sliding window on the same virtual clock
class TokenBucket:
    def __init__(self, capacity, refill_rate_per_sec):
        self.capacity = capacity
        self.tokens = float(capacity)
        self.refill_rate = refill_rate_per_sec
        self.last_time = 0.0

    def allow(self, now):
        elapsed = now - self.last_time
        self.tokens = min(self.capacity, self.tokens + elapsed * self.refill_rate)
        self.last_time = now
        if self.tokens >= 1:
            self.tokens -= 1
            return True, self.tokens
        return False, self.tokens


class SlidingWindowLimiter:
    def __init__(self, max_requests, window_sec):
        self.max_requests = max_requests
        self.window_sec = window_sec
        self.timestamps = []

    def allow(self, now):
        self.timestamps = [stamp for stamp in self.timestamps if now - stamp < self.window_sec]
        if len(self.timestamps) < self.max_requests:
            self.timestamps.append(now)
            return True
        return False


burst_times = np.concatenate([np.linspace(0, 1, 15), np.linspace(3, 5, 4), np.linspace(6, 6.8, 10)])
bucket = TokenBucket(capacity=10, refill_rate_per_sec=4)
window = SlidingWindowLimiter(max_requests=10, window_sec=2)
burst_rows = []
for now in burst_times:
    bucket_allowed, tokens_left = bucket.allow(float(now))
    window_allowed = window.allow(float(now))
    burst_rows.append({"time": now, "bucket_allowed": bucket_allowed, "window_allowed": window_allowed, "tokens_left": tokens_left})
burst_df = pd.DataFrame(burst_rows)

steady_times = np.arange(0, 10, 0.5)
configs = {
    "too tight": (2, 1),
    "configured": (10, 4),
    "too loose": (50, 4),
}
health_rows = []
for label, (capacity, refill) in configs.items():
    steady_bucket = TokenBucket(capacity, refill)
    burst_bucket = TokenBucket(capacity, refill)
    steady_admitted = sum(steady_bucket.allow(float(now))[0] for now in steady_times)
    first_burst_admitted = sum(burst_bucket.allow(float(now))[0] for now in burst_times[burst_times <= 1])
    health_rows.append({
        "config": label,
        "steady_admitted": steady_admitted,
        "steady_total": len(steady_times),
        "burst_admitted": first_burst_admitted,
        "burst_total": int((burst_times <= 1).sum()),
    })
health_df = pd.DataFrame(health_rows)
print(health_df.to_string(index=False))
print("\nTwo-sided check: too tight rejects ordinary traffic; too loose admits the entire burst.")

In [ ]:
# Part 3 - Visualize the two-sided limiter health check
heat = health_df.assign(
    steady_rate=lambda frame: frame["steady_admitted"] / frame["steady_total"],
    burst_rate=lambda frame: frame["burst_admitted"] / frame["burst_total"],
).set_index("config")[["steady_rate", "burst_rate"]]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=axes[0])
axes[0].set_title("Measured admission rates")
axes[0].set_xlabel("traffic pattern")
axes[0].set_ylabel("configuration")
axes[1].plot(burst_df["time"], burst_df["tokens_left"], marker="o", color="#1d4ed8")
axes[1].scatter(
    burst_df.loc[~burst_df["bucket_allowed"], "time"],
    burst_df.loc[~burst_df["bucket_allowed"], "tokens_left"],
    color="#b91c1c",
    label="rejected",
)
axes[1].set_title("Configured token bucket over virtual time")
axes[1].set_xlabel("seconds")
axes[1].set_ylabel("tokens after decision")
axes[1].legend()
plt.tight_layout()
plt.show()

### Request limits are not spending limits

**Predict:** Riverside reserves $0.006 for a request. Actual usage costs $0.004. Does the ledger spend the estimate or refund the difference? What if actual cost is $0.007?

In [ ]:
# Part 3 - Reserve estimated cost atomically, then settle actual cost
class BudgetExceeded(Exception):
    pass


class BudgetLedger:
    def __init__(self, limit_usd):
        self.limit_usd = limit_usd
        self.spent_usd = 0.0
        self.reservations = {}

    @property
    def reserved_usd(self):
        return sum(self.reservations.values())

    @property
    def available_usd(self):
        return self.limit_usd - self.spent_usd - self.reserved_usd

    def reserve(self, request_id, estimate_usd):
        if request_id in self.reservations:
            raise ValueError(f"duplicate reservation: {request_id}")
        if estimate_usd > self.available_usd + 1e-12:
            raise BudgetExceeded(f"need USD {estimate_usd:.4f}, available USD {self.available_usd:.4f}")
        self.reservations[request_id] = estimate_usd

    def settle(self, request_id, actual_usd):
        if request_id not in self.reservations:
            raise KeyError(f"missing reservation: {request_id}")
        estimate = self.reservations.pop(request_id)
        self.spent_usd += actual_usd
        return actual_usd - estimate


ledger = BudgetLedger(limit_usd=0.020)
ledger.reserve("refund-case", 0.006)
refund_delta = ledger.settle("refund-case", 0.004)
ledger.reserve("overage-case", 0.006)
overage_delta = ledger.settle("overage-case", 0.007)

print(f"refund-case settlement delta:  USD {refund_delta:+.3f}")
print(f"overage-case settlement delta: USD {overage_delta:+.3f}")
print(f"actual spent:                  USD {ledger.spent_usd:.3f}")
print(f"remaining available:           USD {ledger.available_usd:.3f}")
print("Payoff: the ledger records actual cost while reservations prevent concurrent overspend.")

## 4. Exact cache first; unsafe semantic cache as public failure

An exact key is safe only if it names every state dimension that can change the answer. Prompt-only caching silently serves the old chapter count after Riverside updates the manuscript.

In [ ]:
# Part 4 - Prove prompt-only keys go stale and versioned exact keys do not
def exact_cache_key(request, model_id, temperature=0.0):
    material = {
        "tenant": request.tenant_id,
        "prompt": " ".join(request.prompt.split()),
        "model_id": model_id,
        "temperature": temperature,
        "corpus_version": request.corpus_version,
        "policy_version": request.policy_version,
    }
    return sha256(json.dumps(material, sort_keys=True).encode("utf-8")).hexdigest()


prompt_only_cache = {RIVERSIDE_PROMPT: "There are 21 chapters."}
request_v1 = GatewayRequest("cache-v1", "riverside", RIVERSIDE_PROMPT, 9, 32, corpus_version="corpus-v1")
request_v2 = GatewayRequest("cache-v2", "riverside", RIVERSIDE_PROMPT, 9, 32, corpus_version="corpus-v2")
versioned_cache = {
    exact_cache_key(request_v1, "fast-v1"): "There are 21 chapters.",
    exact_cache_key(request_v2, "fast-v1"): "There are 23 chapters.",
}

stale_answer = prompt_only_cache[request_v2.prompt]
fresh_answer = versioned_cache[exact_cache_key(request_v2, "fast-v1")]
print(f"prompt-only key after corpus update: {stale_answer}")
print(f"versioned key after corpus update:   {fresh_answer}")
print(f"stale prompt-only result detected:   {stale_answer != fresh_answer}")

In [ ]:
# Part 4 - Measure exact-cache latency and cost savings on repeated traffic
cache_traffic = [
    "What is RAG?",
    "What is fine-tuning?",
    "What is RAG?",
    "What is an LLM gateway?",
    "What is RAG?",
    "What is fine-tuning?",
]
cache_store = {}
cache_rows = []
provider_cost = 0.00012
provider_latency = 180.0
for index, prompt in enumerate(cache_traffic):
    key = ("riverside", prompt, "fast-v1", "corpus-v2", "policy-v3")
    hit = key in cache_store
    if not hit:
        cache_store[key] = f"answer-{index}"
    cache_rows.append({
        "request": index,
        "cache_hit": hit,
        "latency_ms": 2.0 if hit else provider_latency,
        "cost_usd": 0.0 if hit else provider_cost,
        "no_cache_cost_usd": provider_cost,
    })
cache_evidence = pd.DataFrame(cache_rows)
hit_rate = cache_evidence["cache_hit"].mean()
cache_savings = 1 - cache_evidence["cost_usd"].sum() / cache_evidence["no_cache_cost_usd"].sum()
print(cache_evidence.to_string(index=False))
print(f"\nexact-cache hit rate: {hit_rate:.1%}")
print(f"measured cost savings: {cache_savings:.1%}")
print(f"mean latency: {cache_evidence['latency_ms'].mean():.1f} ms")

### Attempt 4: semantic similarity is not semantic equivalence

A toy lexical cache makes the danger visible. Production embeddings are stronger than word overlap, but they do not remove the need for authorization, versioned keys, exclusions, and thresholds calibrated on false reuse.

In [ ]:
# Part 4 - Publish the unsafe semantic-cache failure around negation
def word_overlap_similarity(left, right):
    def words(text):
        return set(re.findall(r"[a-z0-9']+", text.lower()))
    left_words, right_words = words(left), words(right)
    return len(left_words & right_words) / len(left_words | right_words)


cached_prompt = "How many chapters does The Cartographer's Cipher have?"
paraphrase = "How many chapters are in The Cartographer's Cipher?"
negated = "How many chapters does The Cartographer's Cipher not have?"
threshold = 0.50
semantic_scores = pd.Series({
    "true paraphrase": word_overlap_similarity(cached_prompt, paraphrase),
    "meaning changed by not": word_overlap_similarity(cached_prompt, negated),
})

axis = semantic_scores.plot.bar(color=["#15803d", "#b91c1c"], figsize=(7.5, 3.8))
axis.axhline(threshold, color="#b45309", linestyle="--", label=f"threshold={threshold:.2f}")
axis.set_ylim(0, 1)
axis.set_ylabel("toy lexical similarity")
axis.set_title("Attempt 1 fails: overlap cannot protect meaning")
axis.legend()
plt.tight_layout()
plt.show()

print(semantic_scores.round(3).to_string())
print(f"\nparaphrase would hit: {semantic_scores['true paraphrase'] >= threshold}")
print(f"negated prompt would also hit: {semantic_scores['meaning changed by not'] >= threshold}")
print("Complaint: similarity alone is neither authorization nor semantic equivalence.")

## 5. Retry the same route; fallback to another approved route

The recovery policy needs a stable error taxonomy. This lab uses:

- \`timeout\` and \`throttled\`: retryable;
- \`invalid_request\` and \`policy_denied\`: not retryable.

One overall deadline and per-attempt timeouts are production requirements. The deterministic lab measures attempt count and route changes without sleeping.

In [ ]:
# Part 5 - Record every attempt and enforce retry-before-fallback semantics
@dataclass
class AttemptRecord:
    request_id: str
    attempt: int
    route: str
    kind: str
    status: str
    error_code: str | None
    retryable: bool | None
    latency_ms: float
    cost_usd: float


def execute_with_recovery(request, route_chain, retries_per_route=1):
    attempts = []
    attempt_number = 0
    for route_index, route in enumerate(route_chain):
        for local_attempt in range(retries_per_route + 1):
            attempt_number += 1
            kind = "initial" if attempt_number == 1 else ("retry" if local_attempt > 0 else "fallback")
            try:
                response = route.adapter.complete(request)
                attempts.append(AttemptRecord(
                    request.request_id, attempt_number, route.name, kind, "ok", None, None,
                    response.latency_ms, response.cost_usd,
                ))
                return response, attempts
            except GatewayError as error:
                attempts.append(AttemptRecord(
                    request.request_id, attempt_number, route.name, kind, "error",
                    error.code, error.retryable, 0.0, 0.0,
                ))
                if not error.retryable:
                    return None, attempts
                if local_attempt < retries_per_route:
                    continue
                break
    return None, attempts


timeout = GatewayError("timeout", "simulated timeout", retryable=True)
primary_native = DeterministicNativeProvider(
    "hosted-fast", "openai", [180], [7], failure_plan=[timeout, timeout]
)
fallback_native = DeterministicNativeProvider("hosted-premium", "anthropic", [280], [8])
primary_route = Route(
    "hosted-fast",
    ProviderAdapter("hosted-fast", "fast-v1", primary_native, 0.00015, 0.00060, "openai"),
    "uk", {"chat"}, True, True,
)
fallback_route = Route(
    "hosted-premium",
    ProviderAdapter("hosted-premium", "premium-v3", fallback_native, 0.00080, 0.00240, "anthropic"),
    "uk", {"chat"}, True, True,
)

recovery_request = GatewayRequest("req-recovery", "riverside", RIVERSIDE_PROMPT, 9, 32)
recovered, recovery_attempts = execute_with_recovery(
    recovery_request, [primary_route, fallback_route], retries_per_route=1
)
recovery_df = pd.DataFrame(asdict(attempt) for attempt in recovery_attempts)
print(recovery_df.to_string(index=False))
print(f"\nfinal provider: {recovered.provider}")
print(f"same-route retries: {(recovery_df['kind'] == 'retry').sum()}")
print(f"route-changing fallbacks: {(recovery_df['kind'] == 'fallback').sum()}")

In [ ]:
# Part 5 - Prove non-retryable errors stop and fallback depth has diminishing returns
invalid = GatewayError("invalid_request", "prompt is invalid", retryable=False)
invalid_native = DeterministicNativeProvider(
    "hosted-fast", "openai", [180], [7], failure_plan=[invalid]
)
invalid_route = Route(
    "hosted-fast",
    ProviderAdapter("hosted-fast", "fast-v1", invalid_native, 0.00015, 0.00060, "openai"),
    "uk", {"chat"}, True, True,
)
invalid_response, invalid_attempts = execute_with_recovery(
    GatewayRequest("req-invalid", "riverside", "bad request", 2, 8),
    [invalid_route, fallback_route],
    retries_per_route=1,
)

failure_rates = [0.35, 0.20, 0.02]
depth_rows = []
simulation_rng = random.Random(19)
for depth in [1, 2, 3]:
    theoretical_failure = math.prod(failure_rates[:depth])
    failures = 0
    trials = 10000
    for _ in range(trials):
        failures += all(simulation_rng.random() < rate for rate in failure_rates[:depth])
    depth_rows.append({
        "depth": depth,
        "theoretical_failure_rate": theoretical_failure,
        "observed_failure_rate": failures / trials,
    })
fallback_depth_df = pd.DataFrame(depth_rows)

print(f"non-retryable attempts made: {len(invalid_attempts)}")
print(f"fallback used after invalid input: {any(attempt.kind == 'fallback' for attempt in invalid_attempts)}")
print("\nFallback depth evidence:")
print(fallback_depth_df.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Measured boundary: deeper independent fallback improves availability with diminishing returns and extra work.")

![Animated retry and fallback timeline](images/fallback-chain-retry-timeline.gif)

The animation is a reference artifact; the tables above are the measured source of truth. Real routes may share a region, network, upstream model, or policy dependency, so multiplying independent failure rates can overstate resilience.

**Your turn:** set \`retries_per_route=0\` in the recovery cell. Verify that route changes happen sooner and total attempts fall.

## 6. Compose the taught mechanisms and emit telemetry

Only now is a composed gateway helper readable. It does not hide a new operation; it wires together the operations you already measured.

In [ ]:
# Part 6 - Compose exact cache, limits, budget, eligibility, routing, recovery, and telemetry
class GatewayLab:
    def __init__(self, routes, limiter, ledger):
        self.routes = routes
        self.limiter = limiter
        self.ledger = ledger
        self.cache = {}
        self.request_records = []
        self.attempt_records = []

    def handle(self, request, now, require_hosted=True):
        cache_key = exact_cache_key(request, "routing-alias-v1")
        if cache_key in self.cache:
            cached = self.cache[cache_key]
            response = GatewayResponse(**{**asdict(cached), "cache_status": "exact"})
            self.request_records.append({
                "request_id": request.request_id, "status": "ok", "cache_status": "exact",
                "provider": response.provider, "attempts": 0, "latency_ms": 2.0,
                "cost_usd": 0.0, "rejection_reason": None,
            })
            return response

        estimated_tokens = request.estimated_input_tokens + request.max_output_tokens
        allowed, reason = self.limiter.admit(now, estimated_tokens)
        if not allowed:
            self.request_records.append({
                "request_id": request.request_id, "status": "rejected", "cache_status": "miss",
                "provider": None, "attempts": 0, "latency_ms": 0.0,
                "cost_usd": 0.0, "rejection_reason": reason,
            })
            return None

        approved = eligible_routes(request, self.routes, require_hosted=require_hosted)
        if not approved:
            self.limiter.release()
            raise GatewayError("no_eligible_route", "no route satisfies policy", retryable=False)
        approved = sorted(approved, key=lambda route: (route.in_flight, route.estimated_cost_per_1k))
        estimate_usd = estimated_tokens * approved[0].estimated_cost_per_1k / 1000

        try:
            self.ledger.reserve(request.request_id, estimate_usd)
        except BudgetExceeded:
            self.limiter.release()
            self.request_records.append({
                "request_id": request.request_id, "status": "rejected", "cache_status": "miss",
                "provider": None, "attempts": 0, "latency_ms": 0.0,
                "cost_usd": 0.0, "rejection_reason": "budget",
            })
            return None

        try:
            response, attempts = execute_with_recovery(request, approved, retries_per_route=1)
            self.attempt_records.extend(asdict(attempt) for attempt in attempts)
            actual_cost = sum(attempt.cost_usd for attempt in attempts)
            self.ledger.settle(request.request_id, actual_cost)
            if response is None:
                status, provider, latency = "error", None, sum(attempt.latency_ms for attempt in attempts)
            else:
                status, provider = "ok", response.provider
                latency = sum(attempt.latency_ms for attempt in attempts)
                self.cache[cache_key] = response
            self.request_records.append({
                "request_id": request.request_id, "status": status, "cache_status": "miss",
                "provider": provider, "attempts": len(attempts), "latency_ms": latency,
                "cost_usd": actual_cost, "rejection_reason": None,
            })
            return response
        finally:
            self.limiter.release()


def build_gateway_lab():
    gateway_routes = build_routes()
    return GatewayLab(
        gateway_routes,
        MultiAxisLimiter(rpm=20, tpm=1000, concurrency=3),
        BudgetLedger(limit_usd=0.050),
    )


gateway = build_gateway_lab()
batch_prompts = [
    RIVERSIDE_PROMPT,
    "Summarize chapter one.",
    RIVERSIDE_PROMPT,
    "List the main characters.",
    "Summarize chapter one.",
    "Explain the ending.",
]
for index, prompt in enumerate(batch_prompts):
    gateway.handle(
        GatewayRequest(
            request_id=f"req-{index:02d}",
            tenant_id="riverside",
            prompt=prompt,
            estimated_input_tokens=len(prompt.split()),
            max_output_tokens=32,
        ),
        now=float(index),
    )

request_log = pd.DataFrame(gateway.request_records)
attempt_log = pd.DataFrame(gateway.attempt_records)
print("Request records:")
print(request_log.to_string(index=False))
print("\nAttempt records:")
print(attempt_log.to_string(index=False))

**Code walkthrough**

1. A complete exact key can return before provider-facing limits and budget reservation.
2. A miss estimates tokens, checks RPM/TPM/concurrency, and filters eligibility.
3. Least-busy is the first objective; eligible-set cost breaks ties.
4. Budget reserves the selected route's estimate before recovery starts.
5. Every attempt becomes an attempt record; the request receives one final record.
6. Settlement uses actual attempt cost. A successful approved response becomes cacheable.

The helper is intentionally in-memory. Distributed state, idempotency, authentication, encryption, and output-safety enforcement are production work, not implied by this composition.

In [ ]:
# Part 6 - Report request outcomes, retry amplification, fallback share, latency, and cost
successful = request_log[request_log["status"] == "ok"]
provider_attempts = attempt_log[attempt_log["status"] == "ok"] if not attempt_log.empty else attempt_log
metrics = {
    "requests": len(request_log),
    "success_rate": (request_log["status"] == "ok").mean(),
    "cache_hit_rate": (request_log["cache_status"] == "exact").mean(),
    "mean_attempts_per_request": request_log["attempts"].mean(),
    "fallback_share": (attempt_log["kind"] == "fallback").mean() if not attempt_log.empty else 0.0,
    "p95_latency_ms": float(np.percentile(successful["latency_ms"], 95)),
    "total_cost_usd": request_log["cost_usd"].sum(),
}
for name, value in metrics.items():
    formatted = f"{value:.4f}" if isinstance(value, float) else str(value)
    print(f"{name:<28} {formatted}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
request_log.assign(cumulative_cost=request_log["cost_usd"].cumsum()).plot(
    x="request_id", y="cumulative_cost", marker="o", ax=axes[0], legend=False
)
axes[0].set_title("Cumulative measured cost")
axes[0].set_ylabel("USD")
request_log.plot.bar(x="request_id", y="latency_ms", color="#1d4ed8", ax=axes[1], legend=False)
axes[1].set_title("End-to-end simulated latency")
axes[1].set_ylabel("milliseconds")
plt.tight_layout()
plt.show()

print("Interpretation: exact hits add neither provider cost nor provider attempts and return at 2 ms.")

### Simulation to production bridge

| Lab component | Production equivalent |
|---|---|
| \`ProviderAdapter\` | LiteLLM, provider SDK adapter, or internal inference client |
| In-memory route list | Versioned deployment registry and policy engine |
| In-memory counters | Atomic distributed RPM/TPM/concurrency controls |
| \`BudgetLedger\` | Durable reservation and settlement ledger |
| Dictionary cache | Tenant-isolated shared cache with TTL and invalidation |
| DataFrame records | Metrics, structured logs, and distributed traces |

Popular options remain context-dependent:

| Gateway | Shape | Useful when |
|---|---|---|
| LiteLLM | Open-source library/proxy | Broad provider normalization and self-hosting |
| Cloud/API gateway plus policy | Managed control plane | Identity, networking, policy, and platform integration dominate |
| Custom service | Application-owned | Eligibility and accounting rules are unusually specific |

The lab proves deterministic mechanics, not provider reliability, model quality, cloud policy behavior, or production savings.

In [ ]:
# Part 6 - Preserve environment-driven production configuration and cache namespacing
@dataclass(frozen=True)
class ProductionGatewayConfig:
    primary_model: str
    fallback_models: tuple[str, ...]
    required_secret_envs: tuple[str, ...] = ()
    timeout_seconds: float = 20.0
    max_retries: int = 2
    max_tokens: int = 512
    max_request_cost_usd: float = 0.05
    cache_ttl_seconds: int = 300
    cache_namespace: str = "llm-gateway:v1"
    policy_version: str = "2026-10"

    @classmethod
    def from_environment(cls):
        primary = os.getenv("LLM_GATEWAY_PRIMARY_MODEL", "").strip()
        fallbacks = tuple(
            model.strip()
            for model in os.getenv("LLM_GATEWAY_FALLBACK_MODELS", "").split(",")
            if model.strip()
        )
        secret_names = tuple(
            name.strip()
            for name in os.getenv("LLM_GATEWAY_REQUIRED_SECRET_ENVS", "").split(",")
            if name.strip()
        )
        if not primary:
            raise RuntimeError("Set LLM_GATEWAY_PRIMARY_MODEL to an approved model alias.")
        missing = [name for name in secret_names if not os.getenv(name)]
        if missing:
            raise RuntimeError(f"Required secret environment variables are not configured: {missing}")
        return cls(
            primary_model=primary,
            fallback_models=fallbacks,
            required_secret_envs=secret_names,
        )


def production_cache_key(prompt, tenant_id, config):
    material = {
        "tenant": tenant_id,
        "prompt": " ".join(prompt.split()),
        "primary_model": config.primary_model,
        "max_tokens": config.max_tokens,
        "policy_version": config.policy_version,
    }
    digest = sha256(json.dumps(material, sort_keys=True).encode("utf-8")).hexdigest()
    return f"{config.cache_namespace}:{digest}"


print("Production configuration is read from environment variables, never notebook secrets.")
print("Required secret names are validated only when the explicitly enabled live branch runs.")
print("Production cache keys retain tenant, model alias, generation limit, and policy version.")

**Production bridge walkthrough**

The production configuration preserves the old notebook's guarded source evidence without claiming
a deployment. Approved model aliases, fallbacks, timeouts, retries, token ceilings, cache namespace,
TTL, and policy version are configuration. Secret **names** may be listed; secret values remain in
the environment. The disabled cell below is the only cell allowed to import LiteLLM or make a
network call.

In [ ]:
# Part 6 - Preserve the guarded real-SDK bridge without making a network call by default
RUN_LIVE_LLM_DEMO = os.getenv("RUN_LIVE_LLM_DEMO") == "1"

print("Lab component -> LiteLLM-style production equivalent")
print("ProviderAdapter -> litellm.completion(model=..., messages=...)")
print("GatewayResponse -> normalize text, usage, provider/model, latency, cost")
print("GatewayError -> map SDK exception to stable code + retryable flag")

if RUN_LIVE_LLM_DEMO:
    from litellm import completion

    production_config = ProductionGatewayConfig.from_environment()
    smoke_prompt = os.getenv("LLM_GATEWAY_SMOKE_PROMPT", "Reply with: gateway bridge ready")
    smoke_tenant = os.getenv("LLM_GATEWAY_SMOKE_TENANT", "riverside-smoke")
    print(f"cache key prefix: {production_cache_key(smoke_prompt, smoke_tenant, production_config)[:24]}...")
    call_kwargs = {
        "model": production_config.primary_model,
        "messages": [{"role": "user", "content": smoke_prompt}],
        "max_tokens": production_config.max_tokens,
        "timeout": production_config.timeout_seconds,
        "num_retries": production_config.max_retries,
        "metadata": {"tenant_id": smoke_tenant},
    }
    if production_config.fallback_models:
        call_kwargs["fallbacks"] = list(production_config.fallback_models)
    live_response = completion(
        **call_kwargs,
    )
    print(live_response.choices[0].message.content)
else:
    print("\nLive call skipped. Set RUN_LIVE_LLM_DEMO=1, LLM_GATEWAY_PRIMARY_MODEL,")
    print("optional LLM_GATEWAY_FALLBACK_MODELS / LLM_GATEWAY_REQUIRED_SECRET_ENVS,")
    print("and provider credentials to opt in.")

In [ ]:
# Closing scorecard - Collect every measured mechanism in one place
scorecard = pd.DataFrame([
    {"claim": "adapter contract is stable", "measured": same_contract, "evidence": f"{len(adapter_examples)} provider shapes"},
    {"claim": "eligibility blocked global cheapest", "measured": globally_cheapest.name not in [route.name for route in approved], "evidence": globally_cheapest.name},
    {"claim": "least-busy reduced selected queue", "measured": lb_mean < rr_mean, "evidence": f"{rr_mean:.2f} -> {lb_mean:.2f}"},
    {"claim": "limits reject distinct overloads", "measured": set(limit_evidence.loc[~limit_evidence['allowed'], 'reason']) == {'rpm', 'tpm', 'concurrency'}, "evidence": "RPM + TPM + concurrency"},
    {"claim": "budget settled estimate deltas", "measured": refund_delta < 0 < overage_delta, "evidence": f"{refund_delta:+.3f}, {overage_delta:+.3f}"},
    {"claim": "versioned cache avoided stale answer", "measured": stale_answer != fresh_answer, "evidence": "21 -> 23 chapters"},
    {"claim": "semantic cache exposed unsafe negation", "measured": semantic_scores['meaning changed by not'] >= threshold, "evidence": f"score={semantic_scores['meaning changed by not']:.2f}"},
    {"claim": "retry and fallback stayed distinct", "measured": list(recovery_df['kind']) == ['initial', 'retry', 'fallback'], "evidence": "same route, then new route"},
    {"claim": "composed telemetry recorded requests", "measured": len(request_log) == len(batch_prompts), "evidence": f"{len(request_log)} request records"},
])
print(scorecard.to_string(index=False))

all_passed = bool(scorecard["measured"].all())
print(f"\nAll required measured claims passed: {all_passed}")
if not all_passed:
    raise AssertionError("One or more gateway evidence checks failed.")

## Coverage ledger, decision, and forward boundary

### Built and measured
- Deterministic native providers and three adapters producing one normalized contract.
- Eligibility before round-robin, least-busy, latency-aware EMA, weighted, and cost routing.
- RPM, TPM, concurrency, token bucket, sliding window, and two-sided limiter sizing.
- Atomic-style budget reserve/settle mechanics, including refund and overage.
- Complete exact cache keys, repeated-traffic savings, stale prompt-only failure, and unsafe negation reuse.
- Error-aware same-route retry, approved-route fallback, depth evidence, request telemetry, attempt telemetry, cost, and latency.

### Explained or illustrated
- LiteLLM/provider-SDK mapping, distributed state, redaction, authentication, idempotency, shared cache invalidation, and policy/version rollout.

### Named and intentionally out of scope
- Circuit breakers remain a forward boundary until a measured implementation shows open/half-open/closed behavior.
- Gateway admission protects dispatch. Serving backpressure, continuous batching, KV-cache management, and GPU scheduling remain inside deployments.
- Live provider behavior, cloud identity/networking/policy, quality, and savings remain unvalidated until exercised in the target environment.

**Riverside decision:** ship one normalized contract; version aliases and policies; filter hard constraints before routing; combine provider-facing limits with atomic budget reservation; default to complete exact keys; keep semantic reuse off until false-hit evidence is acceptable; retry only retryable errors within one deadline; fallback only inside the approved set; and retain both request and attempt stories.

Continue into [AI Engineer: Application Latency and Cost](../../role-based-tracks/ai-engineer/03-application-latency-and-cost/README.md) for stage attribution, TTFT/TPOT, retry amplification, and cost denominators, then [Azure Operational LLM Serving](../../ai-infrastructure/09-azure-operational-llm-serving/README.md) for live admission, deadlines, idempotency, and tail-latency failures.